# ERNIE-RNA

In [ ]:
import os
import shutil
import subprocess
import pandas as pd
import time
from pathlib import Path

In [ ]:
method_name = "ERNIE-RNA"
base = Path.cwd()
tools_path = base.parent / "tools"
ernie_rna_dir = tools_path / "ERNIE-RNA"
ernie_rna_env = ernie_rna_dir / "ernie-rna-env"
ernie_python = ernie_rna_env / "bin" / "python"
dataset_name = "bpRNA-1m"
device_id = 0  # GPU id (0, 1, ...); use -1 for CPU

pretrain_ckpt = ernie_rna_dir / "checkpoint" / "ERNIE-RNA_checkpoint" / "ERNIE-RNA_pretrain.pt"
ss_ckpt = (
    ernie_rna_dir
    / "checkpoint"
    / "ERNIE-RNA_ss_prediction_checkpoint"
    / "ERNIE-RNA_attn-map_ss_prediction_bpRNA-1m_checkpoint.pt"
)

print(f"Base directory: {base}")
print(f"ERNIE-RNA directory: {ernie_rna_dir}")
print(f"ERNIE-RNA environment: {ernie_rna_env}")
print(f"ERNIE-RNA python: {ernie_python}")

In [ ]:
# Clone ERNIE-RNA if needed
if not ernie_rna_dir.exists():
    print("Cloning ERNIE-RNA source code...")
    os.makedirs(str(tools_path), exist_ok=True)
    os.chdir(str(tools_path))
    !git clone https://github.com/Bruce-ywj/ERNIE-RNA.git
    os.chdir(str(base))
    print("ERNIE-RNA cloned successfully")
else:
    print(f"ERNIE-RNA already exists at {ernie_rna_dir}")

In [ ]:
if not ernie_rna_env.exists():
    print("Creating ERNIE-RNA conda environment...")
    start = time.time()
    result = subprocess.run(
        [
            "conda", "env", "create",
            "-f", str(ernie_rna_dir / "environment.yml"),
            "--prefix", str(ernie_rna_env),
            "-y",
        ],
        cwd=str(ernie_rna_dir),
    )
    elapsed = time.time() - start
    if result.returncode != 0:
        raise RuntimeError(f"conda env create failed after {elapsed/60:.1f} min")
    print("ERNIE-RNA environment created successfully")
else:
    print(f"ERNIE-RNA environment already exists at {ernie_rna_env}")

subprocess.run(
    [str(ernie_python), "-c", "import fairseq; print(f'fairseq {fairseq.__version__}')"],
    check=True,
)
print(f"ERNIE-RNA environment ready at {ernie_rna_env}")

In [ ]:
# Download pretrained checkpoints if needed (Google Drive links from official repo)
%pip install -q gdown

pretrain_ckpt.parent.mkdir(parents=True, exist_ok=True)
ss_ckpt.parent.mkdir(parents=True, exist_ok=True)

if not pretrain_ckpt.exists():
    print(f"Downloading pretrain checkpoint to {pretrain_ckpt}...")
    !gdown 1CmNxJxgjDhRoBdlDODFFjDNNzJMHVzmO -O {pretrain_ckpt}
else:
    print(f"Pretrain checkpoint already exists at {pretrain_ckpt}")

if not ss_ckpt.exists():
    print(f"Downloading bpRNA-1m SS checkpoint to {ss_ckpt}...")
    !gdown 1tvoHc66uQ796mqlbKgJrl-MiRPQGylR5 -O {ss_ckpt}
else:
    print(f"bpRNA-1m SS checkpoint already exists at {ss_ckpt}")

In [ ]:
def read_virus_fasta(path: str):
    lines = [ln.strip() for ln in open(path, 'r').read().splitlines() if ln.strip() != '']
    records = []
    for i in range(0, len(lines), 3):
        header, seq, struct = lines[i], lines[i+1], lines[i+2]
        name = header[1:].strip()
        records.append((name, seq.strip(), struct.strip()))
    df = pd.DataFrame(records, columns=['name', 'sequence', 'structure']).set_index('name')
    return df

viruses = read_virus_fasta('../data/viruses.fasta')

selected_virus_keys = None

if selected_virus_keys is None:
    virus_ids = list(viruses.index)
else:
    tmp = []
    for k in selected_virus_keys:
        if isinstance(k, int):
            tmp.append(viruses.index[k])
        else:
            tmp.append(str(k))
    virus_ids = tmp

In [ ]:
def run_folding(fasta_name):
    input_path = base / fasta_name
    out_file_name = base / "ERNIE-RNA_folded_seq.fasta"
    save_path = base / "ERNIE-RNA_tmp_output"

    if save_path.exists():
        shutil.rmtree(save_path)
    save_path.mkdir(parents=True, exist_ok=True)

    with open(input_path) as fin:
        seq_id = fin.readline().strip("> \n")
        sequence = fin.readline().strip()

    cmd = [
        str(ernie_python), "predict_ss_rna.py",
        "--dataset_name", dataset_name,
        "--seqs_path", str(input_path),
        "--save_path", str(save_path),
        "--ernie_rna_pretrained_checkpoint", str(pretrain_ckpt),
        "--device", str(device_id),
    ]

    # Must run inside ERNIE-RNA repo: relative paths like ./src/dict/ are used internally.
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(ernie_rna_dir))
    if result.returncode != 0:
        print(f"[ERR] predict_ss_rna.py failed for {seq_id}")
        err = (result.stderr or result.stdout or "").strip()
        if err:
            print(err[-1500:])
        shutil.rmtree(save_path, ignore_errors=True)
        return None

    ct_file = save_path / f"{seq_id}_finetune_prediction.ct"
    if not ct_file.exists():
        ct_candidates = list(save_path.glob(f"{seq_id}*_finetune_prediction.ct"))
        if not ct_candidates:
            print(f"[ERR] No finetune CT file found for {seq_id}")
            shutil.rmtree(save_path, ignore_errors=True)
            return None
        ct_file = ct_candidates[0]

    dot_file = base / f"tmp_{seq_id}.dot"
    convert_cmd = [
        "python", str(base / "ct2dot.py"),
        str(ct_file), str(dot_file),
        "-f", "full", "-q",
    ]
    convert_result = subprocess.run(convert_cmd, capture_output=True, text=True)
    if convert_result.returncode != 0:
        print(f"[ERR] ct2dot.py failed for {seq_id}: {convert_result.stderr[:300]}")
        shutil.rmtree(save_path, ignore_errors=True)
        if dot_file.exists():
            dot_file.unlink()
        return None

    structure = None
    if dot_file.exists():
        with open(dot_file) as f:
            lines = f.readlines()
            if len(lines) >= 3:
                structure = lines[2].strip()

    if dot_file.exists():
        dot_file.unlink()
    shutil.rmtree(save_path, ignore_errors=True)

    if structure is None:
        print(f"[ERR] Failed to extract structure for {seq_id}")
        return None

    with open(out_file_name, "w") as fout:
        fout.write(f">{seq_id}\n")
        fout.write(f"{sequence}\n")
        fout.write(f"{structure}\n")

    return str(out_file_name)

In [ ]:
os.makedirs('../prediction', exist_ok=True)
out_fasta_name = '../prediction/ERNIE-RNA.fasta'
if os.path.exists(out_fasta_name):
    os.remove(out_fasta_name)

print(f"{' ':3}\t{'virus':<20}\t{'len':<5}\t{'time'}")
for i, vid in enumerate(virus_ids):
    start_time = time.time()
    seq = viruses.loc[vid]['sequence']
    print(f"{i+1:3d}/{len(virus_ids)}\t{vid:<20}\t{len(seq):<5}\t", end='', flush=True)

    tmp_fasta = "ERNIE-RNA_tmp.fasta"
    with open(tmp_fasta, "w") as ofile:
        ofile.write(f">{vid}\n{seq}\n")

    dot_file_name = run_folding(tmp_fasta)

    if dot_file_name and os.path.exists(dot_file_name):
        os.system("cat " + dot_file_name + " >> " + out_fasta_name)

    if os.path.exists(tmp_fasta):
        os.remove(tmp_fasta)
    if dot_file_name and os.path.exists(dot_file_name):
        os.remove(dot_file_name)

    elapsed_time = time.time() - start_time
    if dot_file_name:
        print(f"{elapsed_time: .1f} s")
    else:
        print(f"{elapsed_time: .1f} s [fail]")